# Heat Flow Forward Modeling Tutorial

This notebook demonstrates a complete heat flow forward modeling workflow using the `forward-model` library:
1. Define a geological model with thermal properties
2. Compute the surface heat flow anomaly
3. Plot the results
4. Export data

**Physical background:** Surface heat flow anomalies arise from lateral contrasts in thermal
conductivity and/or radiogenic heat production. A granitic intrusion (k ≈ 3.3 W/m·K,
A ≈ 2–4 µW/m³) buried in lower-conductivity sediments produces a positive heat flow
anomaly above it. Continental background heat flow is ~65 mW/m².

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from forward_model import (
    GeologicBody,
    HeatFlowModel,
    ThermalProperties,
    calculate_anomaly,
    plot_combined,
)
from forward_model.io import write_csv, write_json

## Step 1: Define the geological model

In [ ]:
# A granitic intrusion with elevated conductivity and radiogenic heat production
# Depth: 500–2000 m, Width: 1000 m, centred on the profile
granite = GeologicBody(
    name="Granitic Intrusion",
    thermal=ThermalProperties(
        conductivity=3.3,    # W/m·K — granite (high, vs ~1.5 for surrounding shale)
        heat_generation=3.0, # µW/m³ — elevated from radioactive decay of U, Th, K
    ),
    vertices=[
        [-500.0,  500.0],
        [ 500.0,  500.0],
        [ 500.0, 2000.0],
        [-500.0, 2000.0],
    ],
)

observation_x = np.linspace(-3000, 3000, 121).tolist()
model = HeatFlowModel(
    bodies=[granite],
    observation_x=observation_x,
    observation_z=0.0,
    background_heat_flow=65.0,  # mW/m² — continental average
)
print(f"Model has {len(model.bodies)} body, {len(model.observation_x)} observation points")
print(f"Background heat flow: {model.background_heat_flow} mW/m²")

## Step 2: Compute the heat flow anomaly

In [ ]:
# calculate_anomaly dispatches automatically for HeatFlowModel
result = calculate_anomaly(model)

print(f"Heat flow perturbation range: {result.heat_flow.min():.2f} to {result.heat_flow.max():.2f} mW/m²")
print(f"Peak anomaly at x = {observation_x[result.heat_flow.argmax()]:.0f} m")
print(f"Total surface heat flow at peak: {model.background_heat_flow + result.heat_flow.max():.1f} mW/m²")

## Step 3: Plot the results

In [ ]:
fig = plot_combined(
    model,
    result.heat_flow,
    component="heat_flow",
    gradient=result.heat_flow_gradient,
    style="default",
)
plt.show()

## Step 4: Export results

In [ ]:
# Export the vertical heat flow perturbation to CSV
write_csv(
    "heatflow_results.csv",
    model.observation_x,
    result.heat_flow,
    anomaly_label="anomaly_mW_m2",
)
print("Saved heat flow anomaly to heatflow_results.csv")

# Export full model + results to JSON for later visualisation
write_json(
    "heatflow_results.json",
    model,
    result.heat_flow,
    anomaly_label="anomaly_mW_m2",
)
print("Saved model and results to heatflow_results.json")

## Next steps

- Increase `heat_generation` to see how radiogenic production contributes independently
- Try a body with lower conductivity than the host (e.g. 1.2 W/m·K for sedimentary fill) to
  model a heat flow low over a basin
- Explore 2.5D and 2.75D finite-strike extensions using `strike_half_length` on the body
- Run the same model from the CLI: `forward-model run examples/granitic_basin.json`
- See the [theory docs](../docs/theory.md) for the mathematical background